# <center>YOLO 后处理详解</center>

后处理阶段的核心任务是将模型的输出解码为可解释的目标检测结果，同时去除冗余信息，确保最终输出的准确性和可靠性。本节课程将深入探讨 YOLO 后处理中的关键技术，涵盖 **边框还原**、**非极大值抑制（NMS）**，以及**基于 CUDA 的加速实现**。

YOLO 的后处理主要包括以下内容：

- **边框还原**：将检测框坐标还原到原始图像的尺寸。
- **非极大值抑制（NMS）**：去除冗余的边界框，保留最佳检测结果。

通过以上步骤，YOLO 的后处理能够将模型的原始输出转换为高质量的检测结果，从而提升检测性能。

## <center>边框还原</center>

```python
def scale_boxes(img1_shape, boxes, img0_shape, ratio_pad=None):
    """Rescales (xyxy) bounding boxes from img1_shape to img0_shape, optionally using provided `ratio_pad`."""
    if ratio_pad is None:  # calculate from img0_shape
        gain = min(img1_shape[0] / img0_shape[0], img1_shape[1] / img0_shape[1])  # gain  = old / new
        pad = (img1_shape[1] - img0_shape[1] * gain) / 2, (img1_shape[0] - img0_shape[0] * gain) / 2  # wh padding
    else:
        gain = ratio_pad[0][0]
        pad = ratio_pad[1]

    boxes[..., [0, 2]] -= pad[0]  # x padding
    boxes[..., [1, 3]] -= pad[1]  # y padding
    boxes[..., :4] /= gain
    clip_boxes(boxes, img0_shape)
    return boxes
````

在上一节中，我们已经计算出了缩放比例 $r$，以及在宽度和高度方向上的填充量 $dw$ 和 $dh$。假设边界框在模型输入图像中的坐标为 $(x', y')$，那么将其还原到原始图像尺寸的边界框坐标 $(x, y)$ 可以通过以下公式计算：

$$
\begin{aligned}
x = \frac{x' - dw}{r}, y = \frac{y' - dh}{r}
\end{aligned}
$$

此外，我们也可以利用仿射变换矩阵 $M$ 来实现相同的效果。在上一节中，我们已经知道，对于一个点 $(x, y)$ ，其经过变换后的坐标 $(x', y')$ 可以通过以下矩阵运算得到：

$$
\begin{pmatrix}
x' \\
y' \\
\end{pmatrix}
=
\begin{pmatrix}
r & 0 & dw \\
0 & r & dh \\
\end{pmatrix}
\begin{pmatrix}
x \\
y \\
\end{pmatrix}
$$

展开后得到：

$$
\begin{aligned}
x' = r \cdot x + dw, y' = r \cdot y + dh
\end{aligned}
$$

为了将检测框从预处理后的图像尺寸还原到原始图像尺寸，我们需要进行逆变换。逆变换的矩阵为：

$$
M^{-1} = \begin{pmatrix}
\frac{1}{r} & 0 & -\frac{dw}{r} \\
0 & \frac{1}{r} & -\frac{dh}{r}
\end{pmatrix}
$$

通过逆变换公式，我们可以得到：

$$
\begin{aligned}
x = \frac{x' - dw}{r}, y = \frac{y' - dh}{r}
\end{aligned}
$$

## <center>非极大值抑制（NMS）</center>

非极大值抑制（Non-Maximum Suppression, NMS）是目标检测领域中一种关键的后处理技术，主要用于消除冗余的边界框，从而保留最优的检测结果。其核心原理是通过置信度排序，选取置信度最高的边界框，并剔除与其重叠度较高的其他边界框。

**NMS 算法流程**

1. **参数设置**：为置信度阈值和交并比阈值设定初始值。
2. **排序**：按置信度从高到低对边界框进行排序。
3. **筛选**：剔除置信度低于阈值的边界框。
4. **迭代处理**：遍历剩余的边界框，从置信度最高的框开始。
5. **计算交并比**：计算当前框与其余所有框的交并比（IOU），这些框属于同一类别。
6. **剔除冗余框**：如果两个框的 IOU 大于阈值，则剔除置信度较低的框。
7. **重复操作**：重复上述步骤，直到遍历完所有边界框。

在 ultralytics 的源码 [non_max_suppression](https://github.com/ultralytics/ultralytics/blob/v8.3.74/ultralytics/utils/ops.py#L181) 中，NMS 的实现依赖于 `torchvision.ops.nms` 或 `nms_rotated` 来获取非极大值抑制后的索引。特别需要注意的是，`nms_rotated` 专门用于处理旋转边界框（OBB）任务，因为其计算交并比（IOU）的方式与常规检测任务有所不同，能够更好地适应旋转框的几何特性。

在实际应用中，模型的输入通常是批量图像（即 batch 大于 1），且每张图像可能包含多个检测结果。如果逐个处理每个检测结果，会导致计算效率低下。为了显著提升处理速度，可以利用 CUDA 编写核函数进行加速。在这方面，TensorRT 官方已经提供了高效的 [Efficient NMS Plugin](https://github.com/NVIDIA/TensorRT/tree/release/10.7/plugin/efficientNMSPlugin)。该插件通过直接嵌入到 TensorRT 引擎中，减少了 Kernel Launch 的开销，从而大幅提升了处理效率。

基于官方实现的 Efficient NMS 插件，我们进一步进行了优化和扩展，开发了适用于旋转边界框的 [Efficient Rotated NMS](./course_codes/02.02.CustomPlugin/src/efficientRotatedNMSPlugin) 以及支持索引功能的 [Efficient NMS Plugin With Indices](./course_codes/02.02.CustomPlugin/src/efficientIdxNMSPlugin)。这些改进不仅能够满足更复杂的检测需求，还为旋转框任务和高性能场景提供了更高效的解决方案。

## <center>总结</center>

本节课程详细介绍了 YOLO 后处理中的两个关键技术：**边框还原**和**非极大值抑制（NMS）**。通过边框还原，我们可以将模型输出的检测框坐标从预处理后的图像尺寸还原到原始图像尺寸，确保检测结果的准确性。而 NMS 则通过去除冗余的边界框，保留最优的检测结果，进一步提升检测性能。

在实际应用中，为了提高处理效率，我们还探讨了基于 CUDA 的加速实现，特别是 TensorRT 提供的 Efficient NMS 插件及其优化版本。这些技术不仅适用于常规的目标检测任务，还能有效处理旋转边界框等复杂场景。

通过掌握这些后处理技术，我们能够将 YOLO 模型的原始输出转换为高质量的检测结果，从而在实际应用中实现更高效、更准确的目标检测。